# Syria Crop Suitability PoC - Training Notebook

This notebook trains first-screening suitability models for olive and Damask rose.


## 1. Setup


In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(ROOT / 'src'))
ROOT


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from crop_profiles import load_crop_profiles, load_project_config
from suitability import add_all_crop_rule_scores, add_best_crop_columns
from utils import load_feature_columns, read_csv_safely, validate_columns, coerce_numeric, suitability_class


## 2. Load data
Run `src/make_demo_data.py` first if you do not yet have a GEE export.


In [ ]:
input_csv = ROOT / 'data/raw/demo_syria_crop_features.csv'
if not input_csv.exists():
    print('Demo data not found. Run: python src/make_demo_data.py --output data/raw/demo_syria_crop_features.csv --n 5000')
else:
    df = pd.read_csv(input_csv)
    df.head()


## 3. Compute rule scores


In [ ]:
profiles = load_crop_profiles(ROOT / 'config/crop_profiles.yaml')
scored = add_all_crop_rule_scores(df, profiles)
scored[[c for c in scored.columns if 'rule_score' in c]].describe()


## 4. Train models
The CLI script is the recommended training path. In the notebook, use this shell command:


In [ ]:
!python ../src/train.py --input ../data/raw/demo_syria_crop_features.csv --output ../data/processed/crop_suitability_predictions.csv


## 5. Inspect predictions


In [ ]:
pred = pd.read_csv(ROOT / 'data/processed/crop_suitability_predictions.csv')
pred[['olive_model_score','damask_rose_model_score','best_crop','best_crop_score']].head()


In [ ]:
pred['olive_model_score'].hist(bins=30)
plt.title('Olive suitability score')
plt.xlabel('Score')
plt.ylabel('Count')
plt.show()


In [ ]:
pred['damask_rose_model_score'].hist(bins=30)
plt.title('Damask rose suitability score')
plt.xlabel('Score')
plt.ylabel('Count')
plt.show()


## 6. Run dashboard
From project root:

```bash
streamlit run app/streamlit_app.py
```
